In [32]:
import pandas as pd
from sklearn.datasets import load_wine

df = load_wine(as_frame=True)

x = df.data
x = x[x.columns[:7]]

y = df.target

x.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69


In [33]:
y.value_counts()

target
1    71
0    59
2    48
Name: count, dtype: int64

Классы не очень сбалансированы, поэтому используем метрику f1-score

In [34]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=42)

Приведем признаки к одному масштабу

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(xtrain)

xtrain = pd.DataFrame(scaler.transform(xtrain), columns=x.columns)
xtest = pd.DataFrame(scaler.transform(xtest), columns=x.columns)

Обучаем метод опороных векторов (svm) на тренировочных данных и оцениваем качество на тестовых данных

In [36]:
from sklearn.svm import SVC

model = SVC(kernel='linear')
model.fit(xtrain, ytrain)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [37]:
from sklearn.metrics import f1_score

pred = model.predict(xtest)

f1_score(ytest, pred, average='weighted')


0.8890522875816994

У SVM, как и других моделей есть регуляризация. 
Гиперпараметр С регулирует силу регуляризации. Можно посмотреть, как меняется качество модели при изменении С

In [38]:
import numpy as np
from sklearn.model_selection import cross_val_score
for C in np.arange(0.01, 1.1, 0.1):
    print('C:', C, 'score =', cross_val_score(SVC(kernel='linear', C=C), x, y, cv=3, scoring='f1_weighted').mean())

C: 0.01 score = 0.8007977835085888
C: 0.11 score = 0.8813018280615098
C: 0.21000000000000002 score = 0.8983829315052994
C: 0.31000000000000005 score = 0.9040271984570724
C: 0.41000000000000003 score = 0.8981547042711376
C: 0.51 score = 0.9097995167855153
C: 0.6100000000000001 score = 0.915614786294352
C: 0.7100000000000001 score = 0.9147587684891816
C: 0.81 score = 0.9203839689175304
C: 0.91 score = 0.9147219225053939
C: 1.01 score = 0.9147219225053939


Заново обучим модель на xtrain, но подставим новое значение С

In [43]:
model = SVC(kernel='linear', C=0.8)
model.fit(xtrain, ytrain)
pred = model.predict(xtest)
f1_score(ytest, pred, average='weighted')

0.912826474463303

По умолчанию, SVM не умеет предсказывать вероятности. Для этого надо устновить параметр probability=True при объявлении модели.

In [45]:
model.predict_proba(xtest)

AttributeError: This 'SVC' has no attribute 'predict_proba'

Также у SVM можно настраивать и другие гиперпараметры:

*  class_weight - при сильном дисбалансе классов можно ставить значение 'balanced'

*  probability - если требуется предсказывать вероятности классов, то нужно ставить значение True

*  decision_function_shape - по умолчанию 'ovr' (для многоклассовой классификации можно пробовать опцию 'ovo')